# SIGMOD Exp2 Scan-Only Crossover

This notebook uses `htap_scanonly_wkld` and measures only scan-class transactions.
Setup updates and readable-timestamp publication are untimed.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('.').resolve()
HTAP_SIM_DIR = ROOT / 'benches/hash_join/htap_simulation'
sys.path.insert(0, str(HTAP_SIM_DIR))
from bench_script_functions import parse_result

BIN = ROOT / 'target/release/htap_scanonly_wkld'
OUTDIR = ROOT / 'benches/sigmod_exp2_scanonly_crossover/data'
FIGDIR = ROOT / 'benches/sigmod_exp2_scanonly_crossover/figs'
OUTDIR.mkdir(parents=True, exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'warehouse_count': 7,
    'bucket_num': 4096,
    'update_ratio': 0.0001,
    'readable_every': 2,
    'setup_readable_ts': 12,
    'measured_scan_txs': 100,
    'repeat': 5,
    'warmup_runs': 1,
    'trim': 1,
    'timeout_sec': 900,
    'history_values': [0.00, 0.01, 0.02, 0.04, 0.06, 0.08, 0.10],
    'delta_values': [0.00, 0.01, 0.02, 0.04, 0.06, 0.08, 0.10],
}

TABLE_ORDER = ['naive', 'ivmh', 'heap', 'chain', 'par']
BASE_ARGS = [
    '--warehouse-count', str(CONFIG['warehouse_count']),
    '--bucket-num', str(CONFIG['bucket_num']),
    '--update-ratio', str(CONFIG['update_ratio']),
    '--readable-every', str(CONFIG['readable_every']),
    '--setup-readable-ts', str(CONFIG['setup_readable_ts']),
    '--measured-scan-txs', str(CONFIG['measured_scan_txs']),
    '--distinct-history-targets',
    '--distinct-delta-targets',
]

print('ROOT  :', ROOT)
print('BIN   :', BIN)
print('OUTDIR:', OUTDIR)
print('FIGDIR:', FIGDIR)
print('CONFIG:', CONFIG)

In [ ]:
subprocess.run(
    ['cargo', 'build', '--release', '--bin', 'htap_scanonly_wkld'],
    cwd=ROOT,
    check=True,
)
print('Built', BIN)

In [ ]:
def run_checked(args):
    result = subprocess.run(
        [str(x) for x in args],
        cwd=ROOT,
        capture_output=True,
        text=True,
        timeout=CONFIG['timeout_sec'],
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-4000:])
    return result

def aggregate_trial(df):
    if df.empty:
        return pd.DataFrame(columns=['table_type', 'repair_type', 'duration_ms', 'total_ms'])
    out = (
        df.groupby(['table_type', 'repair_type'], dropna=False, as_index=False)['duration_ms']
        .sum()
    )
    out['total_ms'] = out['duration_ms'] / CONFIG['measured_scan_txs']
    return out

def trim_trial_runs(trials):
    if len(trials) <= 2 * CONFIG['trim']:
        return aggregate_trial(pd.concat(trials, ignore_index=True))
    scored = []
    for i, trial in enumerate(trials):
        score = aggregate_trial(trial)['duration_ms'].sum()
        scored.append((score, i))
    scored.sort()
    keep = [i for _, i in scored[CONFIG['trim']:len(scored)-CONFIG['trim']]]
    keep.sort()
    return aggregate_trial(pd.concat([trials[i] for i in keep], ignore_index=True))

def run_single_sweep(sweep_type, ratio):
    rows = []
    for table_type in TABLE_ORDER:
        args = [
            str(BIN),
            *BASE_ARGS,
            '--sweep-type', sweep_type,
            '--sweep-ratio', str(ratio),
            '--table-type', table_type,
        ]
        print('Running', sweep_type, ratio, 'table=', table_type)
        for _ in range(CONFIG['warmup_runs']):
            run_checked(args)
        trials = []
        for trial in range(CONFIG['repeat']):
            result = run_checked(args)
            trials.append(parse_result(result.stdout, table_type))
        df = trim_trial_runs(trials)
        rows.append(df)
    return pd.concat(rows, ignore_index=True)

def pretty_label(table_type, repair_type):
    base = {
        'naive': 'SNAP',
        'ivmh': 'IVMH',
        'heap': 'MONO',
        'chain': 'DUAL',
        'par': 'EPOCH',
    }[table_type]
    if repair_type in ['', None, 'No Repair']:
        return base
    suffix = {
        'Read Repair': ' (RR)',
        'Write Repair': ' (WR)',
    }.get(repair_type, f' ({repair_type})')
    return base + suffix

def plot_sweep(df, x_col, xlabel, title, stem):
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for (table_type, repair_type), group in df.groupby(['table_type', 'repair_type'], dropna=False):
        group = group.sort_values(x_col)
        ax.plot(group[x_col] * 100.0, group['total_ms'], marker='o', label=pretty_label(table_type, repair_type))
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Latency (ms/tx)')
    ax.set_title(title)
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))
    fig.tight_layout()
    fig.savefig(FIGDIR / f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(FIGDIR / f'{stem}.png', dpi=200, bbox_inches='tight')
    plt.show()


In [ ]:
history_frames = []
for value in CONFIG['history_values']:
    df = run_single_sweep('history', value)
    df['history_ratio'] = value
    history_frames.append(df)
df_history = pd.concat(history_frames, ignore_index=True)
history_csv = OUTDIR / 'scanonly_history.csv'
df_history.to_csv(history_csv, index=False)
display(df_history)
plot_sweep(df_history, 'history_ratio', 'Historical Scan (%)', 'Scan-Only Historical Sweep', 'scanonly-history')
print('Saved', history_csv)

In [ ]:
delta_frames = []
for value in CONFIG['delta_values']:
    df = run_single_sweep('delta', value)
    df['delta_ratio'] = value
    delta_frames.append(df)
df_delta = pd.concat(delta_frames, ignore_index=True)
delta_csv = OUTDIR / 'scanonly_delta.csv'
df_delta.to_csv(delta_csv, index=False)
display(df_delta)
plot_sweep(df_delta, 'delta_ratio', 'Delta Scan (%)', 'Scan-Only Delta Sweep', 'scanonly-delta')
print('Saved', delta_csv)